In [1]:
"""
LNG Cargo Optimization — Multi-Agent Workflow with MCP
=======================================================
Vitol Commercial Desk — Senior MLE Interview Example

Same agent logic as before. MCP replaces all mock data and local tools.

MCP Servers used:
  - internal_ctrm      → Vitol's live positions, obligations, terminal slots
  - platts_mcp         → LNG spot prices, netbacks, JKM/TTF/PEG benchmarks
  - bloomberg_mcp      → market news, price context, analyst targets
  - vessels_mcp        → ship positions, ETA, Q-Flex availability and rates

In dev/testing: swap any MCP server for the mock fallback at the bottom.

Install:
    pip install anthropic
"""

import json
import re
import anthropic
from datetime import datetime

with open("config.json", "r") as f:
    config = json.load(f)

API_KEY = config["API_KEY"]
MODEL = config["MODEL"]
client  = anthropic.Anthropic(api_key=API_KEY)


# ── MCP Server registry ───────────────────────────────────────────────────────
# In production: each URL is an internal or vendor-hosted MCP endpoint.
# In dev: comment out and use MOCK_MCP_SERVERS below.

MCP_SERVERS = [
    {
        "type": "url",
        "name": "internal_ctrm",
        "url":  "https://ctrm.vitol.internal/mcp",   # Vitol's trade/position system
    },
    {
        "type": "url",
        "name": "platts_mcp",
        "url":  "https://mcp.spglobal.com/commodities",  # S&P Platts prices
    },
    {
        "type": "url",
        "name": "bloomberg_mcp",
        "url":  "https://mcp.bloomberg.com/terminal",    # Bloomberg market data
    },
    {
        "type": "url",
        "name": "vessels_mcp",
        "url":  "https://mcp.kpler.com/vessels",         # Kpler vessel tracking
    },
]

# ── Cargo being optimized (trader input, not from MCP) ────────────────────────

CARGO = {
    "origin":               "RasGas T3 (Qatar)",
    "volume_mmbtu":         3_400_000,
    "lng_cost_per_mmbtu":   8.20,
    "cargo_date":           "2024-01-18",
}


# ── Shared helpers ────────────────────────────────────────────────────────────

def call_claude_mcp(system: str, user_message: str, servers: list = None) -> anthropic.types.Message:
    """
    Claude call with MCP servers attached.
    Claude auto-discovers available tools from each MCP server
    and calls them as needed — no manual tool schema required.
    """
    return client.messages.create(
        model=      MODEL,
        max_tokens= 1500,
        system=     system,
        messages=   [{"role": "user", "content": user_message}],
        mcp_servers=servers or MCP_SERVERS,
    )

def call_claude(system: str, user_message: str) -> anthropic.types.Message:
    """Plain Claude call — no MCP. Used for routing and evaluation."""
    return client.messages.create(
        model=    MODEL,
        max_tokens=1500,
        system=   system,
        messages= [{"role": "user", "content": user_message}],
    )

def get_text(response) -> str:
    return "\n".join(b.text for b in response.content if b.type == "text")

def get_mcp_results(response) -> list:
    """Extract all MCP tool results from a response."""
    return [
        b for b in response.content
        if b.type == "mcp_tool_result"
    ]

def parse_json(raw: str) -> dict:
    match = re.search(r'\{.*\}', raw, re.DOTALL)
    return json.loads(match.group()) if match else {}

def parse_json_list(raw: str) -> list:
    match = re.search(r'\[.*\]', raw, re.DOTALL)
    return json.loads(match.group()) if match else []

def print_agent(name: str, content: str):
    print(f"\n{'─'*65}")
    print(f"  🤖 {name}")
    print(f"{'─'*65}")
    print(content)


# ── Specialist Agents (MCP-powered) ──────────────────────────────────────────

def intent_agent(query: str) -> dict:
    """
    Parse trader query — no MCP needed, pure reasoning.
    """
    response = call_claude(
        system="""You are an intent classifier for an LNG trading desk.
Extract from the trader's query:
- cargo_origin: where the LNG is coming from
- cargo_date: the laycan / availability date (YYYY-MM-DD if possible)
- query_type: one of [cargo_optimization, market_check, position_check]
- regions_of_interest: list of regions mentioned, or ["all"] if none specified
- urgency: high / medium / low

Return JSON only: {
  "cargo_origin": "...",
  "cargo_date": "...",
  "query_type": "...",
  "regions_of_interest": [...],
  "urgency": "...",
  "summary": "one line of what trader wants"
}""",
        user_message=query
    )
    result = parse_json(get_text(response))
    print_agent("Intent Agent", json.dumps(result, indent=2))
    return result


def terminal_agent(regions: list, cargo_date: str) -> str:
    """
    Check terminal slot availability via internal CTRM MCP server.
    Claude calls get_terminal_slots() and get_vitol_obligations()
    from the CTRM MCP automatically.
    """
    response = call_claude_mcp(
        system="""You are a terminal operations analyst.
Use the CTRM system tools to:
1. Check available terminal slots for the cargo date and nearby dates
2. Check existing Vitol obligations at each terminal
3. Return a summary of which terminals are open and conflict-free

Be specific — list terminal names, slot dates, and any obligation conflicts.""",
        user_message=f"""Find available LNG terminal slots for:
Cargo origin: {CARGO['origin']}
Cargo date:   {cargo_date}
Regions:      {', '.join(regions)}

Check slot availability and flag any terminals where Vitol already has cargo booked.""",
        servers=[s for s in MCP_SERVERS if s["name"] == "internal_ctrm"]
    )
    result = get_text(response)
    print_agent("Terminal Agent (CTRM MCP)", result)
    return result


def spread_agent(terminals_summary: str, cargo_date: str) -> str:
    """
    Pull live LNG prices and calculate netbacks via Platts MCP.
    Claude calls get_lng_price(), get_netback(), get_spark_spread()
    from the Platts MCP automatically.
    """
    response = call_claude_mcp(
        system="""You are a commodity pricing analyst at an LNG trading house.
Use the Platts pricing tools to:
1. Get current JKM, TTF, and PEG prices
2. Calculate netback for each available terminal
   Netback = benchmark price - LNG cost - freight - regasification
3. Rank terminals by netback margin, highest first

LNG cargo cost: $8.20/MMBtu
Show the netback breakdown per terminal clearly.""",
        user_message=f"""Calculate netbacks for these terminals on {cargo_date}:

{terminals_summary}

Cargo cost: ${CARGO['lng_cost_per_mmbtu']}/MMBtu
Cargo volume: {CARGO['volume_mmbtu']:,} MMBtu""",
        servers=[s for s in MCP_SERVERS if s["name"] == "platts_mcp"]
    )
    result = get_text(response)
    print_agent("Spread Agent (Platts MCP)", result)
    return result


def market_agent(query: str, terminals_summary: str) -> str:
    """
    Pull market context and news via Bloomberg MCP.
    Claude calls get_news(), get_analyst_targets(), get_market_snapshot()
    from Bloomberg MCP automatically.
    """
    response = call_claude_mcp(
        system="""You are a senior commodity market analyst.
Use Bloomberg tools to pull:
1. Latest LNG/gas market news relevant to routing decisions
2. Current supply/demand signals for Asia vs Europe
3. Any analyst price target changes in the last 48 hours

Summarise in 4-5 bullet points. Focus on what affects the cargo routing decision.
Be direct — this is for a trader who needs to act in 30 minutes.""",
        user_message=f"""Trader query: {query}

Available terminal destinations:
{terminals_summary}

What is the current market context that should influence where we send this cargo?""",
        servers=[s for s in MCP_SERVERS if s["name"] == "bloomberg_mcp"]
    )
    result = get_text(response)
    print_agent("Market Agent (Bloomberg MCP)", result)
    return result


def shipping_agent(terminals_summary: str, cargo_date: str) -> str:
    """
    Check vessel availability and freight rates via Kpler MCP.
    Claude calls get_vessel_positions(), get_freight_rate(), get_eta()
    from the Kpler MCP automatically.
    """
    response = call_claude_mcp(
        system="""You are a shipping analyst for an LNG trading desk.
Use the vessel tracking tools to:
1. Find available Q-Flex vessels near Qatar on the cargo date
2. Get current freight rates for each destination route
3. Flag any route risks (canal delays, congestion, weather)

Return freight cost per MMBtu per destination and a risk flag: low/medium/high.""",
        user_message=f"""Cargo: {CARGO['volume_mmbtu']:,} MMBtu Q-Flex from {CARGO['origin']}
Laycan: {cargo_date}

Possible destinations:
{terminals_summary}

What vessels are available, what are the freight rates, and are there any route risks?""",
        servers=[s for s in MCP_SERVERS if s["name"] == "vessels_mcp"]
    )
    result = get_text(response)
    print_agent("Shipping Agent (Kpler MCP)", result)
    return result


def ranking_agent(spread_analysis: str, shipping_analysis: str, market_context: str) -> str:
    """
    Combine all agent outputs and rank destinations.
    No MCP needed — pure reasoning over structured context.
    """
    response = call_claude(
        system="""You are a cargo optimization analyst at a major LNG trading house.
Rank the top 3 terminal destinations combining:
1. Net margin per MMBtu (most important)
2. Shipping cost and risk
3. Market conditions (demand signals, price direction)
4. Obligation conflicts

Return your ranking as:
#1 [terminal] — $X.XX/MMBtu net — [one line rationale]
#2 [terminal] — $X.XX/MMBtu net — [one line rationale]
#3 [terminal] — $X.XX/MMBtu net — [one line rationale]""",
        user_message=f"""Spread and netback analysis:
{spread_analysis}

Shipping analysis:
{shipping_analysis}

Market context:
{market_context}

Rank the top 3 destinations."""
    )
    result = get_text(response)
    print_agent("Ranking Agent", result)
    return result


def recommendation_agent(query: str, context: dict) -> str:
    """Final trader-facing recommendation — no MCP needed."""
    response = call_claude(
        system="""You are a senior LNG analyst briefing a trader.
Be direct and specific. Format exactly like this:

RECOMMENDATION: [terminal name]
MARGIN: $X.XX/MMBtu | Total: $X,XXX,XXX

TOP 3 OPTIONS:
1. [terminal] — $X.XX/MMBtu — [one line reason]
2. [terminal] — $X.XX/MMBtu — [one line reason]
3. [terminal] — $X.XX/MMBtu — [one line reason]

KEY CALL:
[2-3 lines — why top pick wins, main risk to watch]

NEXT STEP:
[One specific action the trader should take now]

No waffle. Trader needs to act in 30 minutes.""",
        user_message=f"""Trader query: {query}

Cargo: {CARGO['volume_mmbtu']:,} MMBtu from {CARGO['origin']}, laycan {CARGO['cargo_date']}

Terminal availability:
{context.get('terminals', '')}

Netback analysis:
{context.get('spreads', '')}

Shipping:
{context.get('shipping', '')}

Market context:
{context.get('market', '')}

Ranking:
{context.get('ranking', '')}"""
    )
    result = get_text(response)
    print_agent("Recommendation Agent", result)
    return result


def evaluator_agent(query: str, recommendation: str) -> dict:
    """Score the recommendation — no MCP needed."""
    response = call_claude(
        system="""You are a senior LNG trading desk evaluator.
Score this AI-generated cargo recommendation on:
- margin_accuracy: did it identify the highest-margin destination? (1-5)
- risk_awareness: did it flag shipping and market risks? (1-5)
- actionability: can the trader act on this immediately? (1-5)
- clarity: would a busy trader understand this in 10 seconds? (1-5)
- overall: overall quality (1-5)

Return JSON only: {
  "margin_accuracy": N,
  "risk_awareness": N,
  "actionability": N,
  "clarity": N,
  "overall": N,
  "improvement": "one specific thing that would make this better"
}""",
        user_message=f"Trader query: {query}\n\nRecommendation:\n{recommendation}"
    )
    result = parse_json(get_text(response))
    print_agent("Evaluator Agent", json.dumps(result, indent=2))
    return result


# ── Dynamic Router ────────────────────────────────────────────────────────────

def route(intent: dict) -> list:
    """
    Build pipeline from intent — pure Python, no extra Claude call.
    Fast and predictable for a latency-sensitive trading desk.
    """
    query_type = intent.get("query_type", "cargo_optimization")

    if query_type == "market_check":
        return ["market", "recommendation", "evaluator"]

    if query_type == "position_check":
        return ["terminal", "spread", "recommendation", "evaluator"]

    # Full cargo optimization
    return ["terminal", "spread", "market", "shipping", "ranking", "recommendation", "evaluator"]


# ── Orchestrator ──────────────────────────────────────────────────────────────

def orchestrator(trader_query: str) -> dict:
    print(f"\n{'═'*65}")
    print(f"  TRADER QUERY : {trader_query}")
    print(f"  TIME         : {datetime.now().strftime('%Y-%m-%d %H:%M UTC')}")
    print(f"{'═'*65}")

    context = {}

    # Step 1: Always classify intent first (no MCP)
    intent = intent_agent(trader_query)
    context["intent"] = intent
    regions    = intent.get("regions_of_interest", ["all"])
    cargo_date = intent.get("cargo_date", CARGO["cargo_date"])

    # Step 2: Build pipeline from intent
    pipeline = route(intent)
    print(f"\n  📋 Pipeline: intent → {' → '.join(pipeline)}")

    # Step 3: Run each agent, accumulating context
    for agent in pipeline:

        if agent == "terminal":
            context["terminals"] = terminal_agent(regions, cargo_date)

        elif agent == "spread":
            context["spreads"] = spread_agent(context.get("terminals", ""), cargo_date)

        elif agent == "market":
            context["market"] = market_agent(trader_query, context.get("terminals", ""))

        elif agent == "shipping":
            context["shipping"] = shipping_agent(context.get("terminals", ""), cargo_date)

        elif agent == "ranking":
            context["ranking"] = ranking_agent(
                context.get("spreads", ""),
                context.get("shipping", ""),
                context.get("market", "")
            )

        elif agent == "recommendation":
            context["recommendation"] = recommendation_agent(trader_query, context)

        elif agent == "evaluator":
            context["eval"] = evaluator_agent(trader_query, context["recommendation"])

    print(f"\n{'═'*65}")
    print(f"  DONE — Eval score: {context['eval'].get('overall')}/5")
    print(f"{'═'*65}\n")

    return context


# ── Ground truth eval (runs after cargo settles) ──────────────────────────────

def ground_truth_eval(recommendation_log: dict, actual_terminal: str, actual_margin: float):
    """
    Compare recommendation vs actual outcome weeks later.
    Log to database and use to track model improvement over time.
    """
    recommendation  = recommendation_log.get("recommendation", "")
    eval_score      = recommendation_log.get("eval", {})

    print(f"\n{'═'*65}")
    print(f"  GROUND TRUTH EVAL (post-trade settlement)")
    print(f"{'─'*65}")
    print(f"  Actual destination : {actual_terminal}")
    print(f"  Actual margin      : ${actual_margin:.3f}/MMBtu")
    print(f"  LLM eval score     : {eval_score.get('overall')}/5")
    print(f"\n  → Log this to your eval database to track model drift over time.")
    print(f"{'═'*65}\n")

    return {
        "actual_terminal":  actual_terminal,
        "actual_margin":    actual_margin,
        "llm_eval_score":   eval_score.get("overall"),
        "timestamp":        datetime.now().isoformat(),
    }


# ── Dev mode: mock MCP servers for local testing ──────────────────────────────
#
# When MCP servers aren't available (local dev, CI), swap in mock servers
# that return static responses. Agent logic stays identical.
#
# from unittest.mock import patch, MagicMock
#
# def mock_mcp_response(text):
#     mock = MagicMock()
#     mock.content = [MagicMock(type="text", text=text)]
#     return mock
#
# with patch.object(client.messages, "create", return_value=mock_mcp_response(
#     "Gate LNG (Rotterdam): slot 2024-01-18 available, no obligations\n"
#     "Sodegaura (Japan): slot 2024-01-22 available, no obligations"
# )):
#     result = orchestrator("Free Qatar cargo Jan 18, where should it go?")


# ── Run ───────────────────────────────────────────────────────────────────────

if __name__ == "__main__":

    result = orchestrator(
        "I have a free cargo out of Qatar on the 18th, where should it go?"
    )

    # Simulate ground truth eval after cargo settles
    ground_truth_eval(
        recommendation_log=result,
        actual_terminal="Sodegaura (Japan)",
        actual_margin=7.42
    )


═════════════════════════════════════════════════════════════════
  TRADER QUERY : I have a free cargo out of Qatar on the 18th, where should it go?
  TIME         : 2026-06-15 01:41 UTC
═════════════════════════════════════════════════════════════════

─────────────────────────────────────────────────────────────────
  🤖 Intent Agent
─────────────────────────────────────────────────────────────────
{
  "cargo_origin": "Qatar",
  "cargo_date": "2025-01-18",
  "query_type": "cargo_optimization",
  "regions_of_interest": [
    "all"
  ],
  "urgency": "medium",
  "summary": "Trader seeks optimal destination for a free Qatari cargo with laycan on the 18th."
}

  📋 Pipeline: intent → terminal → spread → market → shipping → ranking → recommendation → evaluator


TypeError: Messages.create() got an unexpected keyword argument 'mcp_servers'